<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/7_4_Clustering_K%E2%80%91means%2C_Hierarchical%2C_DBSCAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part VII. Dimensionality Reduction, Unsupervised Learning, and Geometry of Data

## 1. Curse and Blessings of Dimensionality    (Geometry of the d‑dimensional sphere and cube, concentration of measure, condition number of random matrices)
## 2. Principal Component Analysis (PCA) and SVD
## 3. Nonlinear Dimensionality Reduction: t‑SNE, UMAP, Isomap
## 4. Clustering: K‑means, Hierarchical, DBSCAN


## Введение в кластеризацию

Кластеризация — это задача **обучения без учителя**, в которой требуется разбить множество объектов на группы (кластеры) так, чтобы объекты внутри одного кластера были максимально похожи друг на друга, а объекты из разных кластеров — максимально различны. В отличие от классификации, где каждому объекту приписан известный класс, в кластеризации метки отсутствуют, и алгоритм должен самостоятельно обнаружить структуру данных.

Формально, пусть дана выборка $\mathbf{X} = \{\mathbf{x}_1,\dots,\mathbf{x}_n\} \subset \mathbb{R}^d$. Требуется найти разбиение $\mathcal{C} = \{C_1,\dots,C_K\}$, $C_i \subseteq \mathbf{X}$, $C_i \cap C_j = \varnothing$ при $i \neq j$, $\bigcup_i C_i = \mathbf{X}$, которое оптимизирует некоторый критерий качества. Чаще всего стремятся максимизировать **внутрикластерное сходство** и **межкластерное различие**, но конкретный вид критерия зависит от алгоритма.

### Отличие кластеризации от классификации

| Характеристика | Классификация | Кластеризация |
|---------------|---------------|---------------|
| Тип обучения | С учителем | Без учителя |
| Наличие меток | Да (известны заранее) | Нет |
| Цель | Предсказать метку для новых объектов | Найти группы в данных |
| Оценка качества | Точность, F‑мера и т.д. | Внутренние индексы (силуэт, Дэвиса–Болдина), внешние (если есть истина) |
| Пример | Отличить кошек от собак | Выделить породы кошек по генетическим маркерам |

В реальных задачах кластеризация часто предшествует классификации: выделив группы, можно вручную назначить им метки и затем обучить классификатор.

### Примеры применения

- **Сегментация клиентов.** Торговые сети анализируют историю покупок и выделяют группы клиентов со схожим поведением (экономные, премиум, охотники за скидками). Это позволяет персонализировать маркетинговые акции.
- **Сжатие изображений.** Кластеризация цветов пикселей (например, методом K‑means) позволяет сократить палитру изображения с минимальной потерей качества.
- **Обнаружение аномалий.** Объекты, не попавшие ни в один кластер (например, в DBSCAN помечаются как шум), могут быть выбросами — мошенническими транзакциями, дефектами продукции.
- **Биоинформатика.** Кластеризация генов по профилям экспрессии помогает находить группы генов, участвующих в одних и тех же биологических процессах.
- **Рекомендательные системы.** Пользователей со схожими вкусами можно объединить в кластеры и рекомендовать контент, популярный внутри кластера.

### Трудности кластеризации и мотивация разных алгоритмов

Не существует универсального метода кластеризации. Выбор алгоритма диктуется свойствами данных и целями анализа. Рассмотрим ключевые вызовы.

**1. Выбор меры сходства.**  
Евклидово расстояние $d(\mathbf{x},\mathbf{y}) = \sqrt{\sum (x_j - y_j)^2}$ — самая распространённая метрика, но она чувствительна к масштабу признаков и предполагает сферическую форму кластеров. В текстовых данных часто используют косинусное сходство, в сетях — расстояние по графу. Неправильно выбранная метрика может сделать кластеры неразличимыми.

**2. Форма кластеров.**  
Многие реальные группы данных не являются выпуклыми или сферическими. Например, рукописные цифры могут образовывать изогнутые многообразия в пространстве пикселей. Алгоритм K‑means предполагает, что кластеры — это компактные «шары» примерно равного размера. Для кластеров произвольной формы нужны плотностные (DBSCAN) или иерархические методы.

**3. Разная плотность.**  
В данных могут присутствовать как плотные скопления, так и разреженные области. K‑means плохо работает, если кластеры имеют разную плотность или если один кластер находится внутри другого. DBSCAN с правильно подобранными параметрами может выделять кластеры разной плотности (хотя и у него есть ограничения).

**4. Шум и выбросы.**  
Наличие аномальных наблюдений сильно смещает центроиды в K‑means и искажает структуру. DBSCAN изначально проектировался для работы с шумом: точки, не принадлежащие ни одному кластеру, помечаются как «шум». Иерархическая кластеризация также чувствительна к выбросам, но применение устойчивых метрик слияния (метод Уорда) может смягчить эффект.

**5. Неизвестное число кластеров $K$.**  
В отличие от классификации, где количество классов известно, в кластеризации число кластеров редко задано априори. Существуют методы определения $K$: локтевой график, индекс силуэта, статистика разрыва (gap statistic). Некоторые алгоритмы (DBSCAN, иерархическая кластеризация) не требуют указания $K$ в явном виде.

**6. Высокая размерность.**  
При большом числе признаков расстояния между точками становятся почти одинаковыми (проклятие размерности). Перед кластеризацией часто применяют снижение размерности (PCA) или отбор признаков. Некоторые алгоритмы (subspace clustering) специально ищут кластеры в подпространствах.

### Синтетические наборы данных для демонстрации

Чтобы наглядно показать сильные и слабые стороны разных методов, сгенерируем несколько классических двумерных наборов данных.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons, make_circles

# 1. Сферические кластеры (make_blobs)
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, random_state=42, cluster_std=1.5)

# 2. Два полумесяца (make_moons)
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)

# 3. Концентрические окружности (make_circles)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)

# Визуализация
fig, axes = plt.subplots(1, 3, figsize=(15,5))
datasets = [(X_blobs, y_blobs, 'make_blobs\n(сферические кластеры)'),
            (X_moons, y_moons, 'make_moons\n(полумесяцы)'),
            (X_circles, y_circles, 'make_circles\n(вложенные окружности)')]
for ax, (X, y, title) in zip(axes, datasets):
    ax.scatter(X[:,0], X[:,1], c=y, cmap='viridis', edgecolor='k', s=30)
    ax.set_title(title)
plt.tight_layout()
plt.show()
```

Полученные графики иллюстрируют три принципиально разных ситуации:

- **Сферические кластеры (Blobs).** Идеальны для K‑means: центры кластеров хорошо разделены, кластеры компактны и выпуклы.
- **Полумесяцы (Moons).** Кластеры невыпуклые, но чётко разделены. K‑means здесь провалится, так как проведёт границу между кластерами по прямой, не улавливая изогнутую форму. DBSCAN же легко выделит оба полумесяца.
- **Концентрические окружности (Circles).** Один кластер находится внутри другого. K‑means не сможет разделить их, потому что центроид внутреннего кольца и внешнего совпадают по центру масс. DBSCAN при правильно подобранном радиусе сможет отделить кольцо от ядра.

**Данные с шумом.** Добавим к сферическим кластерам случайные выбросы, чтобы показать, как K‑means смещает центроиды.

```python
X_noisy = np.vstack([X_blobs, np.random.uniform(-12, 12, size=(30, 2))])
plt.scatter(X_noisy[:,0], X_noisy[:,1], c='gray', edgecolor='k')
plt.title('Сферические кластеры с выбросами')
plt.show()
```

Присутствие выбросов (30 точек в случайных местах) приведёт к тому, что центроиды K‑means будут утянуты в сторону выбросов, особенно если кластеров мало. DBSCAN пометит такие точки как шум.

### Обзор методов, рассматриваемых в главе

В этой главе мы подробно изучим три фундаментальных подхода к кластеризации, каждый из которых решает описанные выше проблемы по-своему.

1. **K‑means.** Простейший итеративный алгоритм, основанный на минимизации внутрикластерной суммы квадратов расстояний до центров (центроидов). Хорошо работает на сферических кластерах одинакового размера, масштабируется на большие выборки. Требует задания числа кластеров $K$ и чувствителен к начальной инициализации (решается с помощью K‑means++).

2. **Иерархическая кластеризация.** Строит дендрограмму — дерево вложенных разбиений. Позволяет не задавать $K$ заранее: можно выбрать число кластеров, разрезав дендрограмму на нужном уровне. Агломеративные методы (снизу вверх) объединяют ближайшие кластеры на каждом шаге. Варьируя критерий слияния (одиночная, полная, средняя связь, метод Уорда), можно получать кластеры разной формы. Иерархическая кластеризация хорошо визуализируется и интерпретируется, но требует $O(n^3)$ вычислений для наивной реализации (с кучей можно ускорить до $O(n^2)$), что ограничивает её применение на больших данных.

3. **DBSCAN (Density-Based Spatial Clustering of Applications with Noise).** Плотностной алгоритм, который определяет кластеры как области высокой плотности, разделённые областями низкой плотности. Не требует задания $K$, способен находить кластеры произвольной формы и автоматически определяет шум. Ключевые параметры — радиус окрестности `eps` и минимальное число точек в окрестности `min_samples`. DBSCAN хорошо справляется с выбросами, но может испытывать трудности при большой разнице в плотности кластеров.

В последующих разделах мы детально разберём математические основы, реализацию с нуля и применение каждого из этих методов.

### Контрольные вопросы

1. Почему перед кластеризацией важно масштабировать признаки? Приведите пример, когда немасштабированные данные приводят к неправильному разбиению.
2. В каком случае K‑means даст ошибочные результаты? Опишите, как форма кластеров влияет на его работу.
3. Чем DBSCAN принципиально отличается от K‑means с точки зрения понятия «кластер»?
4. Какие метрики можно использовать для оценки качества кластеризации, если истинные метки неизвестны?

В следующем разделе мы начнём с самого известного алгоритма — K‑means. Реализуем его с нуля, изучим свойства и научимся выбирать оптимальное число кластеров.

## K‑means: алгоритм, свойства, выбор числа кластеров

Алгоритм K‑means — один из старейших и наиболее широко используемых методов кластеризации. Он подкупает простотой, интерпретируемостью и вычислительной эффективностью. В основе лежит идея попеременного уточнения центров кластеров (центроидов) и назначения точек ближайшим центрам. Несмотря на эвристичность, K‑means тесно связан с EM-алгоритмом для смеси гауссовских распределений с фиксированными изотропными ковариациями и жёстким присвоением. В этом разделе мы подробно разберём математическую постановку, реализуем алгоритм с нуля, обсудим инициализацию и методы выбора $K$.

### 1. Формальная постановка задачи

Пусть дана выборка $\mathbf{X} = \{\mathbf{x}_1,\dots,\mathbf{x}_n\} \subset \mathbb{R}^d$. Требуется найти набор центроидов $\boldsymbol{\mu} = \{\boldsymbol{\mu}_1,\dots,\boldsymbol{\mu}_K\} \subset \mathbb{R}^d$ и разбиение $\mathcal{C} = \{C_1,\dots,C_K\}$, минимизирующие **внутрикластерную сумму квадратов** (Within-Cluster Sum of Squares, WCSS):

$$
\operatorname{WCSS}(\mathcal{C}, \boldsymbol{\mu}) = \sum_{k=1}^K \sum_{\mathbf{x} \in C_k} \|\mathbf{x} - \boldsymbol{\mu}_k\|^2.
$$

При фиксированных центроидах оптимальное разбиение получается отнесением каждой точки к ближайшему центроиду. При фиксированном разбиении оптимальный центроид каждого кластера — это среднее арифметическое попавших в него точек. Задача минимизации WCSS NP-трудна, и алгоритм K‑means находит лишь локальный оптимум.

### 2. Алгоритм Ллойда (стандартный K‑means)

**Шаг 1. Инициализация.** Выбрать $K$ начальных центроидов $\boldsymbol{\mu}_1^{(0)},\dots,\boldsymbol{\mu}_K^{(0)}$. Традиционный способ — случайно выбрать $K$ точек из данных.

**Шаг 2. Назначение (E-шаг).** Для каждого $\mathbf{x}_i$ найти ближайший центроид:

$$
c_i^{(t)} = \arg\min_{k} \|\mathbf{x}_i - \boldsymbol{\mu}_k^{(t)}\|.
$$

Образуются кластеры $C_k^{(t)} = \{ \mathbf{x}_i \mid c_i^{(t)} = k \}$.

**Шаг 3. Обновление центроидов (M-шаг).** Пересчитать центроиды как средние точек каждого кластера:

$$
\boldsymbol{\mu}_k^{(t+1)} = \frac{1}{|C_k^{(t)}|} \sum_{\mathbf{x} \in C_k^{(t)}} \mathbf{x}.
$$

Если кластер пуст, его центроид обычно переинициализируют (например, случайной точкой из данных).

Шаги 2 и 3 повторяются, пока центроиды изменяются более чем на заданный порог `tol`, или до достижения максимального числа итераций. Критерием сходимости также может служить неизменность разбиения.

Связь с EM-алгоритмом: K‑means можно рассматривать как жёсткий EM для гауссовской смеси с равными весами и ковариационными матрицами $\sigma^2 \mathbf{I}$, где на E-шаге апостериорная вероятность заменяется индикаторной функцией.

### 3. Реализация на NumPy «с нуля»

Напишем векторизованную функцию `kmeans_custom`, использующую евклидовы расстояния и критерий сходимости по изменению центроидов.

```python
import numpy as np
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt

def kmeans_custom(X, K, max_iters=100, tol=1e-4, random_state=None):
    rng = np.random.RandomState(random_state)
    n, d = X.shape
    # Инициализация: случайный выбор K точек
    indices = rng.choice(n, K, replace=False)
    centroids = X[indices].copy()
    labels = np.zeros(n, dtype=int)
    for it in range(max_iters):
        # E-шаг: расстояния до центроидов (n,K)
        dists = np.linalg.norm(X[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2)
        new_labels = np.argmin(dists, axis=1)
        # M-шаг: новые центроиды
        new_centroids = np.array([X[new_labels == k].mean(axis=0) for k in range(K)])
        # Проверка сходимости
        shift = np.linalg.norm(new_centroids - centroids)
        centroids = new_centroids
        if shift < tol:
            break
    # Последнее назначение
    dists = np.linalg.norm(X[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2)
    labels = np.argmin(dists, axis=1)
    return centroids, labels

# Проверка на make_blobs
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, random_state=42, cluster_std=1.5)
centroids, labels = kmeans_custom(X_blobs, K=3, random_state=42)

plt.figure(figsize=(6,5))
plt.scatter(X_blobs[:,0], X_blobs[:,1], c=labels, cmap='viridis', edgecolor='k', s=30)
plt.scatter(centroids[:,0], centroids[:,1], c='red', marker='X', s=200, edgecolor='black')
plt.title('Результат K‑means (собственная реализация)')
plt.show()
```

### 4. Инициализация K‑means++

Случайная инициализация может привести к плохим локальным оптимумам: кластеры могут «склеиться», или один кластер окажется пустым. Алгоритм **K‑means++** (Arthur & Vassilvitskii, 2007) гарантирует, что начальные центроиды распределены по данным более равномерно, что улучшает сходимость и качество.

**Алгоритм K‑means++:**
1. Выбрать первый центроид $\boldsymbol{\mu}_1$ равномерно из $\mathbf{X}$.
2. Для $k = 2$ до $K$:
   - Для каждой точки $\mathbf{x}$ вычислить $D(\mathbf{x})^2$ — квадрат расстояния до ближайшего уже выбранного центроида.
   - Выбрать следующий центроид случайно, с вероятностью, пропорциональной $D(\mathbf{x})^2$.
3. Запустить стандартный K‑means с полученными центроидами.

Реализуем функцию инициализации и сравним качество (финальный WCSS) случайной и K‑means++ инициализации на 30 запусках.

```python
def initialize_kmeans_plus_plus(X, K, random_state=None):
    rng = np.random.RandomState(random_state)
    n = X.shape[0]
    centroids = np.zeros((K, X.shape[1]))
    # первый центроид — случайная точка
    first_idx = rng.choice(n)
    centroids[0] = X[first_idx]
    for k in range(1, K):
        # расстояния до ближайшего центроида
        dists = np.min(np.linalg.norm(X[:, np.newaxis, :] - centroids[:k][np.newaxis, :, :], axis=2), axis=1)
        # вероятности пропорциональны квадрату расстояния
        probs = dists ** 2 / np.sum(dists ** 2)
        next_idx = rng.choice(n, p=probs)
        centroids[k] = X[next_idx]
    return centroids

# Сравнение инициализаций
def run_kmeans(X, K, init_func, n_runs=30):
    wcss_list = []
    for seed in range(n_runs):
        if init_func == 'random':
            rng = np.random.RandomState(seed)
            idx = rng.choice(X.shape[0], K, replace=False)
            init_centroids = X[idx]
        else:
            init_centroids = initialize_kmeans_plus_plus(X, K, random_state=seed)
        # Запускаем kmeans_custom с заданными начальными центроидами
        centroids, labels = kmeans_custom(X, K, random_state=seed) # упрощённо
        # (в нашей функции kmeans_custom инициализация внутри, сделаем вариант с передачей init)
        # Перепишем быструю версию...
    return wcss_list
```

Чтобы не усложнять, воспользуемся готовым `sklearn.cluster.KMeans` с параметром `init='random'` и `init='k-means++'` для сравнения.

```python
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt
import numpy as np

X, _ = make_blobs(n_samples=500, centers=5, random_state=0, cluster_std=1.2)
wcss_random = []
wcss_pp = []
for seed in range(30):
    km_random = KMeans(n_clusters=5, init='random', n_init=1, max_iter=300, random_state=seed)
    km_random.fit(X)
    wcss_random.append(km_random.inertia_)
    km_pp = KMeans(n_clusters=5, init='k-means++', n_init=1, max_iter=300, random_state=seed)
    km_pp.fit(X)
    wcss_pp.append(km_pp.inertia_)

plt.boxplot([wcss_random, wcss_pp], labels=['random', 'k-means++'])
plt.ylabel('WCSS')
plt.title('Сравнение инициализаций (30 запусков)')
plt.show()
```

График показывает, что K‑means++ систематически даёт меньший разброс и более низкий WCSS, избегая откровенно плохих локальных минимумов.

### 5. Выбор числа кластеров $K$

Один из главных практических вопросов — сколько кластеров выбрать? Рассмотрим два наиболее популярных метода.

**Метод локтя (elbow).**  
Запускаем K‑means для $K=1,2,\dots,K_{\max}$, вычисляем WCSS. Строим график зависимости WCSS от $K$. Ищем «локоть» — точку, после которой снижение WCSS замедляется, становясь почти линейным. Эта точка считается оптимальным $K$. Метод субъективен: иногда локоть выражен слабо.

**Силуэтный коэффициент.**  
Для каждого объекта $\mathbf{x}_i$ определяем:
- $a(i)$ — среднее расстояние до других точек своего кластера (внутрикластерная компактность),
- $b(i)$ — минимальное среднее расстояние до точек другого кластера (межкластерная разделимость).

Силуэтный коэффициент $s(i) = \frac{b(i)-a(i)}{\max(a(i),b(i))}$ лежит в $[-1,1]$. Значения, близкие к 1, означают, что точка хорошо отнесена к своему кластеру. Средний силуэт по всем точкам характеризует качество разбиения. Выбираем $K$, дающее наибольший средний силуэт.

Реализуем оба подхода для синтетических данных и `iris`.

```python
from sklearn.metrics import silhouette_score
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

# Данные iris
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)

K_range = range(2, 11)
wcss = []
silhouettes = []
for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_iris)
    wcss.append(km.inertia_)
    silhouettes.append(silhouette_score(X_iris, labels))

fig, ax1 = plt.subplots(figsize=(8,5))
ax2 = ax1.twinx()
ax1.plot(K_range, wcss, 'bo-', label='WCSS')
ax1.set_xlabel('Число кластеров K')
ax1.set_ylabel('WCSS', color='b')
ax2.plot(K_range, silhouettes, 'rs-', label='Средний силуэт')
ax2.set_ylabel('Силуэт', color='r')
plt.title('Выбор K для Iris: локоть и силуэт')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.show()
```

Для Iris оба метода указывают на $K=2$ или $K=3$ (реальное число видов — 3). Силуэтный коэффициент обычно более объективен.

Другие индексы: **Калински–Харабаса** (отношение межкластерной дисперсии к внутрикластерной, максимизируется), **Дэвиса–Болдина** (минимизируется). В `sklearn.metrics` есть все основные.

### 6. Предварительная обработка данных

K‑means чувствителен к масштабу признаков, поскольку использует евклидово расстояние. Если один признак измеряется в тысячах, а другой в единицах, первый будет полностью доминировать в расстоянии, и кластеры вытянутся вдоль него. **Стандартизация** (`StandardScaler`) или хотя бы приведение к интервалу (`MinMaxScaler`) — обязательный шаг перед K‑means.

Демонстрация:

```python
# Признаки с сильно различающимся масштабом
X_unscaled = np.hstack([X[:,:1]*100, X[:,1:]])  # первый признак умножаем на 100
# Без масштабирования
km_raw = KMeans(n_clusters=3, random_state=0).fit(X_unscaled)
# Со стандартизацией
X_scaled = StandardScaler().fit_transform(X_unscaled)
km_scaled = KMeans(n_clusters=3, random_state=0).fit(X_scaled)

fig, axes = plt.subplots(1,2, figsize=(12,5))
axes[0].scatter(X_unscaled[:,0], X_unscaled[:,1], c=km_raw.labels_, cmap='viridis')
axes[0].set_title('Без масштабирования')
axes[1].scatter(X_scaled[:,0], X_scaled[:,1], c=km_scaled.labels_, cmap='viridis')
axes[1].set_title('Со стандартизацией')
plt.show()
```

Без стандартизации кластеры полностью определяются первым признаком и выглядят как вертикальные полосы; после стандартизации восстанавливается истинная сферическая форма.

### 7. Недостатки K‑means

- **Чувствительность к выбросам.** Центроид смещается в сторону выброса, увеличивая WCSS. Можно предварительно удалять выбросы или использовать более робастную версию (K‑medoids, K‑medians).
- **Предположение сферической формы и одинакового размера.** На `make_moons` K‑means даёт неверное разбиение, разрезая полумесяцы. Это демонстрирует необходимость других алгоритмов.
- **Непригодность для категориальных признаков.** Расстояние не определено для неметрических шкал. Для таких данных существуют K‑modes, K‑prototypes.
- **Локальные оптимумы.** Алгоритм сходится к локальному минимуму WCSS. Запуск с несколькими начальными приближениями (`n_init=10` в sklearn) помогает найти лучшее решение.

### 8. Ускорение и масштабируемость

Стандартный K‑means имеет сложность $O(n \cdot K \cdot d \cdot \text{iter})$, что приемлемо для больших $n$. Дальнейшее ускорение достигается:
- **Mini‑Batch K‑means** — на каждом шаге обновляет центроиды по небольшой случайной подвыборке (`sklearn.cluster.MiniBatchKMeans`). Подходит для миллионов точек.
- **Алгоритм Элькана** — использует неравенство треугольника для сокращения вычислений расстояний, особенно эффективен при $d$ до нескольких десятков (`algorithm='elkan'` в sklearn).

### 9. Резюме

K‑means — быстрый и понятный алгоритм, который должен быть в арсенале каждого аналитика. Однако его применение оправдано только после масштабирования признаков и при условии, что кластеры примерно сферические и равного размера. Для выбора $K$ рекомендуется комбинировать локоть и силуэт, а также учитывать предметную интерпретируемость. В следующем разделе мы обратимся к иерархической кластеризации, которая не требует заранее заданного числа кластеров и позволяет визуализировать структуру данных в виде дендрограммы.

**Задания для читателя:**
1. Реализуйте K‑means с расстоянием Чебышёва ($\max_j |x_{ij} - \mu_{kj}|$) и проверьте, как изменится форма кластеров на `make_blobs`.
2. Для набора `load_wine()` подберите $K$ с помощью силуэтного коэффициента и сравните с истинным числом классов. Согласуются ли они?
3. Модифицируйте собственную реализацию K‑means так, чтобы она выводила WCSS на каждой итерации. Постройте график сходимости WCSS.

### Иерархическая кластеризация: агломеративный подход, дендрограмма, выбор метода linkage

В отличие от K‑means, иерархическая кластеризация не требует заранее заданного числа кластеров. Она строит **дерево вложенных разбиений** — дендрограмму, которая наглядно показывает структуру данных на разных уровнях детализации. Такой подход особенно полезен на этапе разведочного анализа: мы можем получить кластеры разного масштаба, просто «разрезав» дерево на определённой высоте. Существует два основных типа иерархической кластеризации: **агломеративный** (снизу вверх) и **дивизивный** (сверху вниз). Первый распространён гораздо шире, и мы сосредоточимся именно на нём.

#### 1. Агломеративный алгоритм

Агломеративная кластеризация начинает с тривиального разбиения, где каждый объект является отдельным кластером, и на каждом шаге объединяет два наиболее близких кластера в один. Процесс повторяется, пока все объекты не сольются в один кластер, образуя полное бинарное дерево.

Формально алгоритм выглядит так:

1. Начать с $n$ кластеров: $C_i = \{\mathbf{x}_i\}$, $i=1,\dots,n$.
2. Вычислить матрицу попарных расстояний между кластерами согласно выбранному **критерию сцепления** (linkage).
3. Найти пару кластеров $(C_i, C_j)$ с минимальным расстоянием и объединить их: $C_{new} = C_i \cup C_j$.
4. Обновить матрицу расстояний: вычислить расстояния от нового кластера до всех остальных.
5. Повторять шаги 3–4, пока не останется один кластер.

Результатом является последовательность слияний, которая представляется **дендрограммой** — древовидным графиком, где высота ветвей соответствует расстоянию, на котором произошло объединение.

Вычислительная сложность наивной реализации — $O(n^3)$, так как на каждом из $n-1$ шагов требуется пересчитывать расстояния до нового кластера. С использованием приоритетной очереди сложность можно снизить до $O(n^2 \log n)$, что всё равно ограничивает применение выборками порядка $10^4$ объектов.

#### 2. Критерии сцепления (linkage criteria)

Ключевой момент — как измерить расстояние между кластерами. Разные критерии порождают разные структуры дендрограмм и финальных разбиений.

**Single linkage (одиночная связь, метод ближайшего соседа).** Расстояние между кластерами равно минимуму попарных расстояний между их элементами:
$$
d_{\text{single}}(C_i, C_j) = \min_{\mathbf{x}\in C_i,\;\mathbf{y}\in C_j} \|\mathbf{x} - \mathbf{y}\|.
$$
Этот критерий склонен образовывать вытянутые цепочечные кластеры и страдает от эффекта «шумового моста»: достаточно одной близкой пары точек на границе, чтобы кластеры объединились.

**Complete linkage (полная связь, метод дальнего соседа).** Расстояние берётся как максимум попарных расстояний:
$$
d_{\text{complete}}(C_i, C_j) = \max_{\mathbf{x}\in C_i,\;\mathbf{y}\in C_j} \|\mathbf{x} - \mathbf{y}\|.
$$
Это даёт компактные кластеры примерно равного диаметра, но чувствителен к выбросам.

**Average linkage (средняя связь).** Расстояние усредняется по всем парам:
$$
d_{\text{average}}(C_i, C_j) = \frac{1}{|C_i||C_j|}\sum_{\mathbf{x}\in C_i}\sum_{\mathbf{y}\in C_j} \|\mathbf{x} - \mathbf{y}\|.
$$
Компромиссный вариант, менее чувствительный к шуму, чем single и complete.

**Ward linkage (метод Уорда).** Минимизирует прирост внутрикластерной суммы квадратов (WCSS) при слиянии. Расстояние между кластерами можно выразить через центры:
$$
d_{\text{Ward}}(C_i, C_j) = \sqrt{\frac{2|C_i||C_j|}{|C_i|+|C_j|}} \;\|\boldsymbol{\mu}_i - \boldsymbol{\mu}_j\|,
$$
где $\boldsymbol{\mu}_i$ — центроид $C_i$. Ward даёт сферические кластеры схожего размера, напоминая K‑means, но в детерминированной иерархической форме.

#### 3. Реализация с помощью SciPy

В Python иерархическая кластеризация доступна через `scipy.cluster.hierarchy`. Основные функции:

- `linkage(X, method='ward')` — выполняет агломеративную кластеризацию и возвращает матрицу слияний $Z$ размером $(n-1)\times 4$. Каждая строка содержит индексы объединённых кластеров и расстояние слияния.
- `dendrogram(Z)` — строит дендрограмму.
- `fcluster(Z, t, criterion='maxclust')` — режет дерево для получения плоского разбиения: `t` — число кластеров, либо `t` — порог расстояния при `criterion='distance'`.

Применим алгоритм к набору Iris и построим дендрограмму.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# Загрузка и стандартизация
iris = load_iris()
X = StandardScaler().fit_transform(iris.data)

# Агломеративная кластеризация, метод Уорда
Z = linkage(X, method='ward')

# Дендрограмма
plt.figure(figsize=(10, 6))
dendrogram(Z, labels=iris.target, color_threshold=7)
plt.title('Дендрограмма Iris (Ward)')
plt.xlabel('Индекс образца')
plt.ylabel('Расстояние слияния')
plt.axhline(y=7, color='gray', linestyle='--', label='Порог ≈7')
plt.legend()
plt.show()

# Выделение трёх кластеров
labels_3 = fcluster(Z, t=3, criterion='maxclust')
print("Распределение по кластерам (3):", np.bincount(labels_3))
```

Дендрограмма показывает, что расстояние слияния резко увеличивается на высоте примерно 7, что соответствует переходу от трёх кластеров к двум. Это визуальный критерий выбора $K$.

#### 4. Выбор порога отсечения

Чтобы превратить дерево в плоское разбиение, нужно выбрать высоту разреза. Два основных подхода:

- **Визуальный:** найти самый длинный горизонтальный отрезок без слияний. В дендрограмме Iris это интервал примерно от 3 до 7 — он указывает на три естественных кластера.
- **Количественный:** рассчитать разницу между последовательными высотами слияний (`np.diff(Z[:,2])`) и найти максимальный скачок. Скачок соответствует уровню, где слияние кластеров становится значительно более «дорогим».

Для Iris максимальный скачок между второй и третьей итерациями с конца подтверждает $K=3$.

#### 5. Сравнение методов linkage на синтетических данных

Сгенерируем три эталонных набора (blobs, moons, circles) и применим четыре основных критерия сцепления. Визуализируем разбиения на $K=2$ (или $K=3$ для blobs) и прокомментируем результаты.

```python
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler

datasets = {
    'Blobs': make_blobs(n_samples=300, centers=3, random_state=42, cluster_std=1.2),
    'Moons': make_moons(n_samples=300, noise=0.08, random_state=42),
    'Circles': make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)
}

methods = ['single', 'complete', 'average', 'ward']
fig, axes = plt.subplots(len(datasets), len(methods), figsize=(14, 10))
for i, (name, (X, y)) in enumerate(datasets.items()):
    X_std = StandardScaler().fit_transform(X)
    for j, method in enumerate(methods):
        Z = linkage(X_std, method=method)
        if name == 'Blobs':
            labels = fcluster(Z, t=3, criterion='maxclust')
        else:
            labels = fcluster(Z, t=2, criterion='maxclust')
        axes[i, j].scatter(X[:,0], X[:,1], c=labels, cmap='viridis', s=15, edgecolor='none')
        axes[i, j].set_title(f'{method}')
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
        if j == 0:
            axes[i, j].set_ylabel(name)
plt.tight_layout()
plt.show()
```

**Анализ результатов:**
- **Single linkage** отлично выделяет цепочечные структуры (moons, circles), но на blobs образует вытянутые кластеры, часто цепляя шум.
- **Complete linkage** даёт компактные шары, хорошо работает на blobs, но на moons и circles разрезает фигуры на две примерно равные части.
- **Average linkage** занимает промежуточное положение, на moons может показать адекватное разделение.
- **Ward linkage** похож на K‑means: идеален для blobs, но проваливается на невыпуклых формах — moons и circles он не может правильно разделить, так как стремится к сферическим кластерам равного размера.

Вывод: выбор метода linkage критичен и должен соответствовать геометрии ожидаемых кластеров.

#### 6. Достоинства и недостатки

**Плюсы иерархической кластеризации:**
- Не требуется заранее знать $K$; дендрограмма даёт представление о структуре данных на разных масштабах.
- Можно использовать любую метрику расстояния, в том числе неевклидову (например, косинусное расстояние для текстов).
- Детерминированность (при отсутствии равных расстояний слияния) — в отличие от K‑means с его случайной инициализацией, результат воспроизводим.
- Наглядная визуализация в виде дендрограммы.

**Минусы:**
- Высокая вычислительная сложность ($O(n^2 \log n)$), что делает метод непригодным для $n > 10^4$–$10^5$.
- Чувствительность к шуму и выбросам: одиночный удалённый объект может сильно исказить структуру дерева, особенно при single linkage.
- Результат сильно зависит от выбранного критерия сцепления; необходимо экспериментировать с разными методами.
- Неэффективен для кластеров невыпуклой формы (кроме single linkage, который, однако, страдает от цепочечного эффекта).

#### 7. Дивизивная кластеризация (DIANA)

Дивизивный подход начинает со всех объектов в одном кластере и рекурсивно разделяет его на два. На каждом шаге выбирается кластер с наибольшим диаметром и разделяется так, чтобы максимизировать расстояние между полученными подкластерами. Алгоритм DIANA (DIvisive ANAlysis) работает за $O(2^n)$ в худшем случае, что делает его непрактичным для больших данных. На практике дивизивные методы почти полностью вытеснены агломеративными, поэтому мы ограничиваемся упоминанием.

#### 8. Заключительные замечания

Иерархическая кластеризация — мощный инструмент разведочного анализа, позволяющий «увидеть» естественные группировки без их жёсткого задания. Для данных умеренной размерности (до нескольких тысяч объектов) она часто предпочтительнее K‑means, так как не требует априорного знания $K$ и даёт более детальную картину. Однако для больших выборок или кластеров сложной формы лучше обратиться к масштабируемым методам вроде DBSCAN, рассматриваемого в следующем разделе.

**Задания для самопроверки:**
1. Постройте дендрограмму для набора `fetch_olivetti_faces()` (400 лиц, 4096 пикселей), используя косинусное расстояние. Сколько кластеров выделяется визуально?
2. Объясните, почему single linkage часто образует «цепочки» кластеров, и в каких практических задачах это может быть полезно (например, выделение континуумов).
3. Сравните средний силуэт разбиений, полученных иерархической кластеризацией (Ward) и K‑means на наборе `load_wine()`. Какой метод даёт более компактные кластеры?

## DBSCAN: плотностная кластеризация, параметры, обнаружение аномалий

Алгоритмы, основанные на центроидах (K‑means) или на иерархическом слиянии, хорошо работают, когда кластеры имеют примерно сферическую форму и схожий размер. Однако в реальных данных группы могут быть вытянутыми, изогнутыми, иметь разную плотность и содержать шум. **DBSCAN** (Density‑Based Spatial Clustering of Applications with Noise), предложенный Эстером, Кригелем, Сандером и Сюй в 1996 году, реализует принципиально иной взгляд на кластеризацию: кластеры — это области высокой плотности, разделённые областями низкой плотности. Такой подход позволяет находить кластеры произвольной формы, автоматически определять их число и выделять аномалии.

### 1. Мотивация для плотностных методов

Рассмотрим уже знакомые наборы данных — два полумесяца (`make_moons`) и концентрические окружности (`make_circles`). И K‑means, и иерархическая кластеризация с критерием Уорда неспособны корректно разделить эти структуры: они проводят границы, не соответствующие истинной форме данных. Причина в том, что эти методы неявно предполагают выпуклость кластеров и основаны на глобальных расстояниях. Плотностные методы, напротив, опираются на локальную плотность: кластер растёт из области высокой плотности до тех пор, пока плотность остаётся выше заданного порога. Это позволяет «огибать» изгибы и разделять вложенные фигуры.

### 2. Основные определения

DBSCAN оперирует несколькими ключевыми понятиями. Пусть задана метрика расстояния (по умолчанию евклидова) и два параметра: радиус окрестности $\varepsilon > 0$ и минимальное число точек `minPts`.

**$\varepsilon$-окрестность точки $\mathbf{p}$:**
$$
N_\varepsilon(\mathbf{p}) = \{ \mathbf{q} \in \mathbf{X} \mid \operatorname{dist}(\mathbf{p},\mathbf{q}) \le \varepsilon \}.
$$

**Ядерная точка (core point).** Точка $\mathbf{p}$ называется *ядерной*, если её $\varepsilon$-окрестность содержит не менее `minPts` точек (включая саму $\mathbf{p}$):
$$
|N_\varepsilon(\mathbf{p})| \ge \text{minPts}.
$$

**Прямая достижимость по плотности.** Точка $\mathbf{q}$ *прямо достижима по плотности* из ядерной точки $\mathbf{p}$, если $\mathbf{q} \in N_\varepsilon(\mathbf{p})$.

**Достижимость по плотности.** Точка $\mathbf{p}$ *достижима по плотности* из точки $\mathbf{q}$, если существует последовательность ядерных точек $\mathbf{p}_1,\dots,\mathbf{p}_m$, где $\mathbf{p}_1 = \mathbf{q}$, $\mathbf{p}_m = \mathbf{p}$, и каждая $\mathbf{p}_{i+1}$ прямо достижима из $\mathbf{p}_i$. Это транзитивное замыкание отношения прямой достижимости.

**Связность по плотности.** Точки $\mathbf{p}$ и $\mathbf{q}$ *связны по плотности*, если существует точка $\mathbf{o}$, из которой обе достижимы по плотности. Это определяет кластер как максимальное множество плотностно-связных точек.

**Граничная точка (border point).** Точка, не являющаяся ядерной, но достижимая из некоторой ядерной точки.

**Шум (noise).** Точка, не принадлежащая ни одному кластеру. Шумовые точки не являются ядерными и не достижимы из ядерных.

Кластер в DBSCAN — это множество точек, содержащее хотя бы одну ядерную точку, замкнутое относительно достижимости по плотности и содержащее все граничные точки, достижимые из ядерных точек кластера.

> **Углублённый взгляд.**  
> Определения выше задают отношение эквивалентности на множестве ядерных точек: ядерная точка $\mathbf{p}$ связана с ядерной точкой $\mathbf{q}$, если существует соединяющая их цепочка ядерных точек, где соседние находятся на расстоянии $\le \varepsilon$. Кластер — это объединение такого класса эквивалентности ядерных точек со всеми граничными точками, лежащими в $\varepsilon$-окрестности хотя бы одной из ядерных точек этого класса.

### 3. Алгоритм DBSCAN

Алгоритм выполняет однократный проход по данным, постепенно наращивая кластеры.

1. Инициализировать все точки как «непосещённые».
2. Для каждой непосещённой точки $\mathbf{p}$:
   - Пометить $\mathbf{p}$ как посещённую.
   - Вычислить $N_\varepsilon(\mathbf{p})$.
   - Если $|N_\varepsilon(\mathbf{p})| < \text{minPts}$, пометить $\mathbf{p}$ как шум (временно; позже она может стать граничной точкой другого кластера).
   - Иначе создать новый кластер $C$ и добавить в него $\mathbf{p}$ и все точки из $N_\varepsilon(\mathbf{p})$. Для каждой точки $\mathbf{q} \in N_\varepsilon(\mathbf{p})$ (кроме $\mathbf{p}$), если она ещё не посещена, пометить как посещённую; если она ядерная, рекурсивно добавить её $\varepsilon$-окрестность в кластер (распространение).
3. Повторять, пока есть непосещённые точки.

Важный момент: граничная точка, сначала помеченная как шум, может быть позже включена в кластер, если будет обнаружена в $\varepsilon$-окрестности ядерной точки.

### 4. Реализация DBSCAN с нуля (учебная версия)

Реализуем упрощённую версию для понимания логики, используя `scipy.spatial.KDTree` для быстрого поиска соседей.

```python
import numpy as np
from scipy.spatial import KDTree
from sklearn.datasets import make_moons, make_circles

def dbscan_custom(X, eps, min_pts):
    n = X.shape[0]
    labels = np.full(n, -1, dtype=int)   # -1 означает шум
    visited = np.zeros(n, dtype=bool)
    tree = KDTree(X)
    cluster_id = 0
    
    for i in range(n):
        if not visited[i]:
            visited[i] = True
            neighbors = tree.query_ball_point(X[i], r=eps)
            if len(neighbors) < min_pts:
                labels[i] = -1            # пока шум
            else:
                # новая ядерная точка -> новый кластер
                labels[i] = cluster_id
                # расширение кластера: все соседи
                seed_set = neighbors[:]
                j = 0
                while j < len(seed_set):
                    q = seed_set[j]
                    if not visited[q]:
                        visited[q] = True
                        q_neighbors = tree.query_ball_point(X[q], r=eps)
                        if len(q_neighbors) >= min_pts:
                            seed_set.extend(q_neighbors)
                    if labels[q] == -1:
                        labels[q] = cluster_id
                    j += 1
                cluster_id += 1
    return labels

# Тест на moons и circles
X_moons, _ = make_moons(n_samples=200, noise=0.05, random_state=42)
labels_moons = dbscan_custom(X_moons, eps=0.2, min_pts=5)

X_circles, _ = make_circles(n_samples=200, noise=0.03, factor=0.5, random_state=42)
labels_circles = dbscan_custom(X_circles, eps=0.2, min_pts=5)

fig, axes = plt.subplots(1, 2, figsize=(10,4))
axes[0].scatter(X_moons[:,0], X_moons[:,1], c=labels_moons, cmap='tab10', s=20)
axes[0].set_title('DBSCAN на make_moons')
axes[1].scatter(X_circles[:,0], X_circles[:,1], c=labels_circles, cmap='tab10', s=20)
axes[1].set_title('DBSCAN на make_circles')
plt.show()
```

Алгоритм идеально выделяет два полумесяца и два кольца, помечая изолированные точки как шум (метка -1). K‑means на этих же данных дал бы неверные разбиения.

### 5. Выбор параметра $\varepsilon$ (epsilon)

Один из главных практических вопросов — как выбрать радиус $\varepsilon$. Стандартный метод — **график $k$-расстояний** ($k$-distance graph). Для каждой точки вычисляется расстояние до её $k$-го ближайшего соседа (где $k = \text{minPts}$). Полученные расстояния сортируются по убыванию и отображаются на графике. В идеальном случае график имеет выраженный «изгиб» (колено), правее которого значения резко возрастают. Этот изгиб соответствует подходящему значению $\varepsilon$.

Реализуем:

```python
from sklearn.neighbors import NearestNeighbors

def plot_k_distance(X, min_pts):
    neigh = NearestNeighbors(n_neighbors=min_pts)
    nbrs = neigh.fit(X)
    distances, _ = nbrs.kneighbors(X)
    k_dist = np.sort(distances[:, -1])[::-1]
    plt.plot(k_dist)
    plt.xlabel('Точки (отсортированы по убыванию расстояния)')
    plt.ylabel(f'Расстояние до {min_pts}-го соседа')
    plt.title('k-distance график')
    plt.grid()
    plt.show()

plot_k_distance(X_moons, min_pts=5)
```

На кривой виден резкий перегиб в районе $\varepsilon \approx 0.2$ — именно это значение мы и использовали.

### 6. Выбор minPts

Параметр `minPts` управляет чувствительностью к шуму и минимальным размером кластера. Эмпирическое правило: $\text{minPts} \ge d+1$, где $d$ — размерность данных. Для зашумлённых данных рекомендуется брать большие значения (10–20), чтобы не создавать ложных кластеров из шума. В двумерных задачах часто используют $\text{minPts} = 4$–$5$.

Слишком маленькое `minPts` делает алгоритм чувствительным к локальным флуктуациям плотности; слишком большое — может «размыть» кластеры или вообще не найти ядерных точек.

### 7. Преимущества DBSCAN

- **Произвольная форма кластеров.** DBSCAN успешно находит невыпуклые кластеры (полумесяцы, спирали, кольца).
- **Устойчивость к выбросам.** Точки, не попадающие в плотные области, явно помечаются как шум.
- **Не требует задания числа кластеров.** Число кластеров определяется данными и параметрами $\varepsilon$, `minPts`.
- **Масштабируемость.** При использовании пространственных индексов (KD‑дерево, Ball‑дерево) сложность алгоритма составляет $O(n \log n)$.
- **Детерминированность.** Результат не зависит от случайной инициализации (в отличие от K‑means).

### 8. Недостатки DBSCAN

- **Разная плотность.** DBSCAN плохо справляется, если кластеры имеют сильно различающуюся плотность. Один большой разреженный кластер может быть сочтён шумом, если $\varepsilon$ настроен по плотному кластеру. Для таких случаев разработан алгоритм **OPTICS** и его современная версия **HDBSCAN**.
- **Чувствительность к масштабу.** Евклидово расстояние требует, чтобы признаки были приведены к одному масштабу. Стандартизация (`StandardScaler`) обязательна.
- **Выбор параметров.** $\varepsilon$ и `minPts` необходимо подбирать, и для этого требуется понимание данных (k‑distance график не всегда даёт чёткое колено).
- **Проклятие размерности.** При $d > 30$ расстояния между точками становятся почти одинаковыми, и понятие плотности размывается. DBSCAN теряет эффективность, и рекомендуется предварительно применять PCA.

### 9. Практические примеры

**Обнаружение аномалий.** Добавим к блобам 5% случайных выбросов и покажем, что DBSCAN помечает их как шум.

```python
from sklearn.datasets import make_blobs

X_blobs, _ = make_blobs(n_samples=300, centers=3, random_state=42, cluster_std=1.0)
noise = np.random.uniform(-10, 15, size=(15, 2))
X_noisy = np.vstack([X_blobs, noise])

labels_noisy = dbscan_custom(StandardScaler().fit_transform(X_noisy), eps=0.5, min_pts=5)

plt.scatter(X_noisy[:,0], X_noisy[:,1], c=labels_noisy, cmap='tab10', s=20)
plt.title('DBSCAN с выбросами (метка -1 — шум)')
plt.show()
```

Выбросы получают метку -1 и не искажают основные кластеры.

**Кластеризация изображений.** DBSCAN можно применять для уменьшения цветовой палитры, кластеризуя пиксели в цветовом пространстве RGB и заменяя шумовые пиксели на ближайший цвет.

### 10. Ускоренные версии и HDBSCAN

Библиотека `sklearn.cluster.DBSCAN` поддерживает алгоритмы `'ball_tree'` и `'kd_tree'` для эффективного поиска соседей. Однако для больших данных и переменной плотности более предпочтителен **HDBSCAN** (Hierarchical DBSCAN). Он строит иерархию плотностных кластеров и автоматически определяет устойчивые кластеры без необходимости задавать $\varepsilon$. Параметр `min_cluster_size` задаёт минимальный размер кластера.

Установка: `pip install hdbscan`. Пример использования:

```python
import hdbscan
clusterer = hdbscan.HDBSCAN(min_cluster_size=10, gen_min_span_tree=True)
labels_hdb = clusterer.fit_predict(X_scaled)
```

HDBSCAN практически полностью вытеснил классический DBSCAN в задачах, где плотность кластеров неоднородна.

### 11. Резюме

DBSCAN — мощный и интуитивно понятный алгоритм, основанный на концепции плотностной связности. Он отлично подходит для данных с кластерами произвольной формы и с шумом, но требует тщательной настройки параметров и масштабирования признаков. Для задач с сильно различающейся плотностью или очень высокой размерностью следует переходить к HDBSCAN или предварительно снижать размерность с помощью PCA/UMAP.

**Задания для самопроверки:**
1. Подберите $\varepsilon$ для `make_circles` с помощью k‑distance графика (`minPts=5`). Совпадает ли визуально определённое колено с оптимальным значением, найденным перебором?
2. Что произойдёт, если $\varepsilon$ слишком маленькое? слишком большое? Продемонстрируйте на синтетических данных.
3. Сгенерируйте три кластера с сильно разной плотностью (один очень плотный, другой средний, третий разреженный). Как поведёт себя DBSCAN? Сравните с HDBSCAN.

В следующем разделе мы подведём итоги всей главы по кластеризации, сравнив алгоритмы по ключевым характеристикам и дав практические рекомендации по их выбору.

## Сравнение алгоритмов кластеризации и метрики качества без учителя

Мы рассмотрели три фундаментальных подхода к кластеризации: K‑means, иерархическую агломеративную и DBSCAN. Каждый из них имеет свои сильные стороны и ограничения. В этом заключительном разделе мы систематически сравним алгоритмы по ключевым характеристикам, разберём внутренние метрики оценки качества, проведём эксперимент на синтетических данных и дадим практические рекомендации по выбору метода и интерпретации результатов.

### 1. Сводная таблица сравнения

| Характеристика               | K‑means                          | Иерархическая (агломеративная)       | DBSCAN                                  |
|-----------------------------|----------------------------------|--------------------------------------|-----------------------------------------|
| **Форма кластеров**         | сферические, равного размера     | любая (зависит от linkage)           | произвольная                            |
| **Чувствительность к шуму** | высокая (выбросы смещают центры) | высокая (одиночные точки искажают дерево) | низкая (шум отбрасывается)              |
| **Необходимость задавать K**| да                               | нет (обрезаем дендрограмму)          | нет (требуются ε и minPts)              |
| **Основные параметры**      | K, стратегия инициализации       | критерий сцепления, метрика          | ε (радиус), minPts (минимальное число точек) |
| **Масштабируемость**        | $O(n \cdot K \cdot d)$           | $O(n^2 \log n)$ – $O(n^3)$           | $O(n \log n)$ с пространственным индексом |
| **Высокая размерность**     | плохо (проклятие размерности)    | плохо                                | очень плохо (расстояния выравниваются)  |
| **Детерминированность**     | нет (зависит от начальной инициализации) | да (при одинаковых distance и linkage) | да (при фиксированных параметрах)       |
| **Обнаружение аномалий**    | нет                              | нет                                  | да (точки с меткой -1)                  |
| **Интерпретируемость**      | центры кластеров как средние     | дендрограмма наглядна                | кластеры как области повышенной плотности |

Поясним ключевые отличия. K‑means требует заданного числа кластеров и неявно предполагает сферическую форму. Иерархическая кластеризация, с одной стороны, не нуждается в предварительном задании $K$ и позволяет визуально выбрать уровень разрезания дендрограммы, но плохо масштабируется на большие данные. DBSCAN не только не требует $K$, но и способен находить кластеры произвольной формы и выбросы, однако критически зависит от выбора $\varepsilon$ и `minPts`. Все три метода чувствительны к масштабу признаков, поэтому обязательна предварительная стандартизация.

### 2. Внутренние метрики оценки качества кластеризации

При отсутствии истинных меток качество кластеризации оценивают по формальным критериям, которые измеряют компактность кластеров и их разделимость. Наиболее употребительны три метрики.

**Силуэтный коэффициент (Silhouette score).**  
Для объекта $\mathbf{x}_i$, принадлежащего кластеру $C$, вычисляют $a(i)$ — среднее расстояние до других объектов своего кластера, и $b(i)$ — минимальное среднее расстояние до объектов другого кластера. Силуэт

$$
s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}.
$$

Значение лежит в $[-1, 1]$: чем ближе к 1, тем лучше объект лежит в своём кластере и далёк от соседнего. Средний силуэт по всем точкам характеризует общее качество разбиения.

**Индекс Калински–Харабаса (Calinski–Harabasz score, или Variance Ratio Criterion).**  
Определяется как отношение межкластерной дисперсии к внутрикластерной:

$$
CH = \frac{\mathrm{tr}(\mathbf{B}_K) / (K-1)}{\mathrm{tr}(\mathbf{W}_K) / (n-K)},
$$

где $\mathbf{B}_K$ — матрица межгруппового рассеяния, $\mathbf{W}_K$ — матрица внутригруппового рассеяния. Чем выше индекс, тем лучше кластеры разделены. Он предпочитает компактные сферические кластеры.

**Индекс Дэвиса–Болдина (Davies–Bouldin score).**  
Для каждого кластера $C_i$ вычисляется мера сходства $R_{ij}$ с кластером $C_j$ как отношение суммы внутрикластерных разбросов к расстоянию между центрами:

$$
R_{ij} = \frac{\bar{d}_i + \bar{d}_j}{\|\boldsymbol{\mu}_i - \boldsymbol{\mu}_j\|},
$$

где $\bar{d}_i$ — среднее расстояние от точек кластера $i$ до его центроида. Индекс DB — это среднее по всем кластерам $\max_{j \neq i} R_{ij}$. Чем он **меньше**, тем лучше (в отличие от силуэта и CH). Хорошее разбиение даёт значения около нуля.

Эти метрики легко вычислить в `sklearn.metrics`. Однако они имеют свои ограничения: например, все они предполагают, что каждый объект принадлежит какому‑то кластеру, и не учитывают шум.

### 3. Почему силуэт не всегда подходит для DBSCAN

DBSCAN помечает выбросы и точки низкой плотности как шум (метка -1). Стандартный силуэт требует, чтобы каждая точка имела принадлежность кластеру. Если просто удалить шум перед оценкой, мы искусственно улучшим показатель, исключив «проблемные» точки. Кроме того, метрики, основанные на центроидах (CH, DB), теряют смысл для невыпуклых кластеров, так как центроид может лежать вне кластера (например, у кольца). Поэтому для DBSCAN разработаны специальные меры, в частности **DBCV (Density-Based Clustering Validation)**, который оценивает качество плотностного кластера на основе минимального расстояния между ядерными точками разных кластеров. Функция `hdbscan.validity.validity_index` позволяет вычислить DBCV (чем выше, тем лучше). В эксперименте мы всё же приведём силуэт для DBSCAN после удаления шума, но с оговоркой, что это завышенная оценка.

### 4. Эксперимент на синтетических наборах

Создадим четыре двумерных набора данных, отражающих разные сценарии:

- **Blobs** (сферические кластеры, хорошо разделены, $K=3$).
- **Moons** (два невыпуклых полумесяца, $K=2$).
- **Circles** (концентрические окружности, $K=2$).
- **Varied** (кластеры разной плотности: один плотный из 300 точек, другой разреженный из 50 точек, $K=2$).

Для каждого набора применим три алгоритма: K‑means ($K$ зададим соответственно), агломеративную кластеризацию (Ward, обрезаем до нужного $K$) и DBSCAN (параметры подобраны вручную по визуальному качеству). Перед кластеризацией масштабируем данные `StandardScaler`. Результат визуализируем и вычисляем метрики.

```python
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# Генерация данных
np.random.seed(42)
X_blobs, y_blobs = make_blobs(n_samples=500, centers=3, cluster_std=1.2, random_state=42)
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)
# Varied: один плотный кластер, второй разреженный
X_dense = np.random.normal(loc=(-2, -2), scale=0.3, size=(300, 2))
X_sparse = np.random.normal(loc=(3, 3), scale=2.0, size=(50, 2))
X_varied = np.vstack([X_dense, X_sparse])

datasets = {
    'Blobs': (X_blobs, 3),
    'Moons': (X_moons, 2),
    'Circles': (X_circles, 2),
    'Varied': (X_varied, 2)
}

# Подготовка фигуры для графиков
fig, axes = plt.subplots(4, 3, figsize=(15, 16))
algorithms = {
    'K‑means': lambda X, k: KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X),
    'Hierarchical (Ward)': lambda X, k: AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X),
    'DBSCAN': lambda X, k: DBSCAN(eps=0.35, min_samples=5).fit_predict(X)  # eps подбираем
}
# Для DBSCAN на разных датасетах лучше задать разные параметры
dbscan_params = {
    'Blobs': {'eps': 0.5, 'min_samples': 5},
    'Moons': {'eps': 0.25, 'min_samples': 5},
    'Circles': {'eps': 0.2, 'min_samples': 5},
    'Varied': {'eps': 0.5, 'min_samples': 5}
}

results = []

for i, (name, (X, true_k)) in enumerate(datasets.items()):
    X_scaled = StandardScaler().fit_transform(X)
    row_metrics = {'Dataset': name}
    for j, (alg_name, algo_func) in enumerate(algorithms.items()):
        if alg_name == 'DBSCAN':
            eps = dbscan_params[name]['eps']
            min_samples = dbscan_params[name]['min_samples']
            labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_scaled)
        elif alg_name == 'K‑means':
            labels = KMeans(n_clusters=true_k, random_state=42, n_init=10).fit_predict(X_scaled)
        else:  # Hierarchical
            labels = AgglomerativeClustering(n_clusters=true_k, linkage='ward').fit_predict(X_scaled)
        
        # Визуализация
        axes[i, j].scatter(X[:,0], X[:,1], c=labels, cmap='tab10', s=15, edgecolor='none')
        axes[i, j].set_title(f'{alg_name} ({name})')
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
        if j == 0:
            axes[i, j].set_ylabel(name)
        
        # Оценка метриками (для DBSCAN убираем шум с предупреждением)
        mask = labels != -1
        if alg_name == 'DBSCAN' and not np.all(mask):
            # если есть шум, используем только кластеризованные точки
            score_sil = silhouette_score(X_scaled[mask], labels[mask]) if np.sum(mask) > 1 else np.nan
            score_ch = calinski_harabasz_score(X_scaled[mask], labels[mask]) if np.sum(mask) > 1 else np.nan
            score_db = davies_bouldin_score(X_scaled[mask], labels[mask]) if np.sum(mask) > 1 else np.nan
        else:
            score_sil = silhouette_score(X_scaled, labels)
            score_ch = calinski_harabasz_score(X_scaled, labels)
            score_db = davies_bouldin_score(X_scaled, labels)
        
        row_metrics[f'{alg_name} Silhouette'] = score_sil
        row_metrics[f'{alg_name} CH'] = score_ch
        row_metrics[f'{alg_name} DB'] = score_db
        
    results.append(row_metrics)

plt.tight_layout()
plt.show()

# Вывод таблицы метрик
df_metrics = pd.DataFrame(results).set_index('Dataset')
df_metrics.round(3)
```

**Результаты эксперимента и анализ.**  
На `Blobs` все три алгоритма работают хорошо: кластеры сферические, хорошо разделены. Метрики высокие. На `Moons` K‑means проводит прямую границу, разрезая полумесяцы, что даёт низкий силуэт. Иерархическая с Ward тоже режет полумесяцы примерно пополам. DBSCAN прекрасно выделяет два изогнутых кластера, и его силуэт близок к 0.5–0.6, но если бы мы оставили шум (метки -1), стандартный силуэт не смог бы быть вычислен. На `Circles` картина аналогична: DBSCAN выделяет оба кольца, остальные методы проводят радиальные границы. На `Varied` K‑means и иерархическая кластеризация выделяют два кластера, примерно соответствующих плотному и разреженному скоплениям, но разреженный кластер имеет большой разброс, и метрики ухудшаются. DBSCAN при `eps=0.5` находит плотный кластер и несколько точек в разреженной области, а большинство разреженных точек помечает как шум: фактически он даёт один кластер и шум. Метрики после удаления шума могут быть высокими, но это введение в заблуждение, потому что мы потеряли целую группу.

### 5. Когда использовать какой метод? (шпаргалка)

- **K‑means** — ваш первый кандидат, если:
  - Данные сферические или умеренно вытянутые, без значительных выбросов.
  - Вы можете указать $K$ (или подберёте локтем/силуэтом).
  - Требуется высокая скорость и масштабируемость ($n$ может быть миллионами, `MiniBatchKMeans` ещё быстрее).
  - Нужна простая интерпретация: центры кластеров = средние объекты.

- **Иерархическая кластеризация** — выбирайте, когда:
  - Число кластеров неизвестно, и вы хотите визуально исследовать дендрограмму.
  - $n$ не превышает нескольких тысяч (до 5000).
  - Важна детерминированность и возможность использовать неевклидовы метрики (например, косинусное расстояние).
  - Требуется понять иерархические связи (например, таксономия).

- **DBSCAN** — применяйте, если:
  - Кластеры имеют произвольную, возможно, невыпуклую форму.
  - В данных присутствует шум, который нужно отделить.
  - Число кластеров неизвестно и не может быть легко оценено.
  - $n$ велико (при использовании пространственных индексов).
  - **Не используйте DBSCAN**, если кластеры имеют сильно разную плотность; в этом случае попробуйте **HDBSCAN** или **OPTICS**.

### 6. Типичные ошибки новичков

1. **Использовать accuracy для оценки кластеризации** — метрика предполагает знание истинных меток.
2. **Считать, что высокий силуэт всегда означает хорошую кластеризацию** — он поощряет компактные сферические кластеры, и может быть обманчиво высок для одного компактного кластера, если остальные точки объявлены шумом.
3. **Не масштабировать признаки** — K‑means, иерархическая и DBSCAN с евклидовой метрикой без стандартизации дают смещённые результаты.
4. **Для DBSCAN использовать евклидову метрику без стандартизации** — то же самое; расстояние становится бессмысленным при разных масштабах.
5. **Игнорировать выбросы в K‑means** — один далёкий объект может увести центроид и разрушить структуру.
6. **Полагаться на одну метрику** — сочетайте несколько индексов и обязательно визуализируйте результат (хотя бы двумерные проекции).

### 7. Код‑сводка для быстрого сравнения

Напишем функцию `evaluate_clustering(X, algorithms_dict, metrics_dict)`, которая принимает стандартизованные данные, словарь алгоритмов (имя → fit_predict функтор) и словарь метрик, и выводит графики и таблицу.

```python
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import pandas as pd
import matplotlib.pyplot as plt

def evaluate_clustering(X, algorithms, metrics, title='', figsize=(12, 4)):
    n_alg = len(algorithms)
    fig, axes = plt.subplots(1, n_alg, figsize=figsize)
    if n_alg == 1:
        axes = [axes]
    results = {}
    for ax, (name, predict_func) in zip(axes, algorithms.items()):
        labels = predict_func(X)
        # Убираем шумовые метки, если есть, для метрик (с предупреждением)
        mask = labels != -1
        scores = {}
        for mname, metric_func in metrics.items():
            try:
                if np.sum(mask) > 1:
                    scores[mname] = metric_func(X[mask], labels[mask])
                else:
                    scores[mname] = np.nan
            except:
                scores[mname] = np.nan
        results[name] = scores
        ax.scatter(X[:,0], X[:,1], c=labels, cmap='tab10', s=10, edgecolor='none')
        ax.set_title(name)
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()
    return pd.DataFrame(results).round(3)

# Пример использования на moons
X_moons_scaled = StandardScaler().fit_transform(X_moons)
algos = {
    'K‑means (k=2)': lambda X: KMeans(2, random_state=0).fit_predict(X),
    'Agglomerative (Ward, k=2)': lambda X: AgglomerativeClustering(2, linkage='ward').fit_predict(X),
    'DBSCAN (eps=0.25, minPts=5)': lambda X: DBSCAN(eps=0.25, min_samples=5).fit_predict(X)
}
metrics = {
    'Silhouette': silhouette_score,
    'CH': calinski_harabasz_score,
    'DB': davies_bouldin_score
}
df_res = evaluate_clustering(X_moons_scaled, algos, metrics, title='Moons')
print(df_res)
```

### 8. Заключительные рекомендации

Выбор алгоритма кластеризации — это компромисс между скоростью, формой кластеров, наличием шума и необходимостью задавать число групп. Начинайте с K‑means для быстрого прототипа, визуализируйте PCA проекции. Если структура выглядит нелинейной, переходите к DBSCAN или HDBSCAN. Иерархическую кластеризацию используйте для небольших выборок, когда нужна дендрограмма. Всегда проверяйте устойчивость результатов с помощью нескольких метрик и визуального анализа. Помните, что кластеризация — это не только алгоритм, но и искусство интерпретации, и ни одна метрика не заменит понимание данных.

**Задания для самопроверки:**

1. Создайте синтетический датасет с двумя кластерами сильно разного размера (500 и 50 точек). Сравните силуэт и индекс Калински–Харабаса при K‑means с $K=2$. Какой индекс более устойчив к дисбалансу?
2. Придумайте данные, где DBSCAN с правильно подобранными параметрами даст высокий средний силуэт (после удаления шума), но кластеризация будет заведомо плохой (например, все точки в одном кластере, а одиночные шумовые выбросы).
3. Реализуйте автоматический подбор $\varepsilon$ и `minPts` для DBSCAN через оптимизацию силуэта (игнорируя шум) на заданной сетке параметров. Примените к `make_circles` и обсудите результат.